# Milestone 2 — local parsing and audited output

Prepare M1 separately with `milestone2_prepare.ipynb`. This notebook consumes a prepared raw snapshot on local disk. The input/gathering, hierarchy/sentences, bibliography/key-map and final processing modules from all four owners are integrated. Local integration checks run real owner APIs; human source/output review and full-corpus acceptance remain pending. No success rate or benchmark is implied by opening this notebook.

The chosen workflow (Danh, plan 10.0) is sequential: validate the corpus once per run; process all versions of one publication; finalize, stage/write/read-back/checkpoint; release its trees. Metrics accumulate from every publication, including blocked/empty/error. No workers before a real 50–100-publication pilot, stage profiles, output comparison and RAM evidence.


In [ ]:
from pathlib import Path
PROJECT_DIR = None  # Auto-detect local source or clone to /content/arXivProject on Colab.
REPOSITORY_URL = 'https://github.com/Hireophant/arXivProject.git'
REPOSITORY_BRANCH = 'master'
DATASET_NAME = 'Danh2'  # Danh / Danh1 / Danh2 ... matching preparationM2.
PAPERS_THIS_RUN = 100  # Maximum publications this visit; all versions stay together.
PREPARATION_DIR = Path('/content/drive/MyDrive/arXivProject/preparationM2')
PREPARED_ARCHIVE = None  # Auto-select the unique dataset ZIP; set a path only if ambiguous.
PREPARED_REMOTE_FILE_ID = ''  # remote_verified publication['archive']['id']; avoids Drive mount visibility delay.
PREPARED_SHA256 = ''          # remote_verified publication['sha256'].
PREPARED_SNAPSHOT_ID = ''     # remote_verified publication['snapshot_id'].
PREPARED_PAPERS_PENDING = 0   # remote_verified publication['pending'].
LOCAL_WORK_DIR = Path('/content/arxiv-m2-work')
RAW_CORPUS_ROOT = LOCAL_WORK_DIR / 'raw' / DATASET_NAME
OUTPUT_ROOT = LOCAL_WORK_DIR / 'parsed' / DATASET_NAME
DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/arXivProject/Milestone2') / DATASET_NAME
STAGE_INPUT = True  # Reuse verified local staging within the same runtime.
ALLOW_PARTIAL_PILOT = False  # True chỉ khi cố ý parse snapshot pilot còn pending.
RUN_PARSER = False
RUN_BENCHMARK = False
RESUME = False
STRICT = True
BACKUP_ENABLED = True
BACKUP_BATCH_SIZE = 50
EXPORT_HANDOFF_TABLES = True  # Export Parquet from this run after parsing and benchmark.
RESTORE_ARCHIVES = []  # Verified parsed ZIP batches, never raw M1 backups.
REFERENCE_REVIEW_PATH = None  # Local JSON; snapshot and fingerprints must match.
CODE_REVISION = None  # None records git HEAD; actual source bytes also enter the cache hash.
RAM_SAMPLE_INTERVAL_SECONDS = 0.1
COLAB_ACCELERATOR_OBSERVED = None  # Set string 'None' ONLY after viewing Colab CPU settings.
result = None
run_root = None


## Set up the project

On Colab, leave `PROJECT_DIR=None` for automatic setup. This cell clones the GitHub repository to `/content/arXivProject` if needed, reuses any existing checkout there, and installs `src/requirements.txt` before importing the parser. For local use, install dependencies in your environment and set `PROJECT_DIR` if the checkout is outside the current directory tree.

Drive mounts automatically when parsing or benchmarking with backup enabled. Parser output stays local until backup. Choose a prepared snapshot covering the intended dataset and all its versions.


In [ ]:
import json
import os
import subprocess
import sys
if sys.version_info < (3, 11):
    raise RuntimeError('Python >=3.11 is required.')
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if PROJECT_DIR is None:
    PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'src/parsing/contracts.py').is_file()), None)
if PROJECT_DIR is None and IN_COLAB:
    PROJECT_DIR = Path('/content/arXivProject')
if PROJECT_DIR is None:
    raise RuntimeError('Set PROJECT_DIR to the local arXivProject checkout.')
PROJECT_DIR = Path(PROJECT_DIR).resolve()
if IN_COLAB and not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPOSITORY_BRANCH, '--single-branch',
                    REPOSITORY_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'src/parsing/contracts.py').is_file():
    raise FileNotFoundError(f'M2 source is missing from {PROJECT_DIR}. Set PROJECT_DIR to a valid checkout or a new Colab path.')
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(PROJECT_DIR / 'src/requirements.txt')], check=True)
src = str(PROJECT_DIR / 'src')
if src not in sys.path:
    sys.path.insert(0, src)
print('Source:', PROJECT_DIR, '| Runtime:', 'Colab' if IN_COLAB else 'local')
from parsing.contracts import ParseConfig, CONTRACT_VERSION
from parsing.pipeline import run, run_with_backup, backup_run_batch, restore_run_batch
from parsing.benchmark import benchmark_run
from parsing.serialization import read_publication
if BACKUP_ENABLED and (RUN_PARSER or RUN_BENCHMARK):
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    elif DRIVE_BACKUP_ROOT.as_posix().startswith('/content/drive/'):
        raise RuntimeError('Local run: set BACKUP_ENABLED=False or select a real local backup directory.')
elif STAGE_INPUT and (RUN_PARSER or RUN_BENCHMARK or PREPARED_ARCHIVE is not None) and IN_COLAB and not PREPARED_REMOTE_FILE_ID:
    from google.colab import drive
    drive.mount('/content/drive')
print('Contract:', CONTRACT_VERSION)
print('Raw local:', RAW_CORPUS_ROOT, 'Output local:', OUTPUT_ROOT)
print('Dataset:', DATASET_NAME, 'Backup:', BACKUP_ENABLED, DRIVE_BACKUP_ROOT)
print('Integrated owner APIs loaded. Inspect diagnostics and finish human source/output review before acceptance.')


## Stage prepared raw and optionally restore verified parsed batches

`stage_prepared_archive` verifies/copies one archive, safely extracts into a new local destination, and never restores M1 itself. The input adapter still performs full local validation at the beginning of each run, including resume. Restore parsed batches into separate fresh children of OUTPUT_ROOT; the next run verifies current input/config/code/review signatures and every artifact before accepting a cached publication. Different or damaged caches are rejected.

Raw + archive + output + staging batches must fit local disk. Local work after the last verified backup receipt may need to be rerun if the runtime disappears. A local checkpoint is not a durable Drive backup.


In [ ]:
if STAGE_INPUT and (RUN_PARSER or RUN_BENCHMARK or PREPARED_ARCHIVE is not None or PREPARED_REMOTE_FILE_ID):
    from preparation.transport import stage_prepared_archive
    if PREPARED_REMOTE_FILE_ID:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import hashlib
        from uuid import uuid4
        if not PREPARED_SHA256 or not PREPARED_SNAPSHOT_ID:
            raise ValueError('Remote archive requires SHA-256 and snapshot ID from remote_verified publication.')
        auth.authenticate_user()
        PREPARED_ARCHIVE = LOCAL_WORK_DIR / 'downloads' / (PREPARED_REMOTE_FILE_ID + '.zip')
        PREPARED_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
        if PREPARED_ARCHIVE.exists():
            with PREPARED_ARCHIVE.open('rb') as stream: cached_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
            if cached_sha != PREPARED_SHA256: PREPARED_ARCHIVE.unlink()
        if not PREPARED_ARCHIVE.exists():
            pending = PREPARED_ARCHIVE.with_name(PREPARED_ARCHIVE.name + '.part-' + uuid4().hex)
            try:
                with pending.open('xb') as stream:
                    downloader = MediaIoBaseDownload(stream, build('drive', 'v3').files().get_media(fileId=PREPARED_REMOTE_FILE_ID))
                    done = False
                    while not done:
                        status, done = downloader.next_chunk()
                        if status: print(f'[downloading_verified] {int(status.progress() * 100)}%', flush=True)
                pending.replace(PREPARED_ARCHIVE)
            except BaseException:
                pending.unlink(missing_ok=True)
                raise
        receipt = {'kind': 'm2-prepared-transport-1.0', 'remote_verified': True, 'sha256': PREPARED_SHA256, 'snapshot_id': PREPARED_SNAPSHOT_ID, 'papers_pending': PREPARED_PAPERS_PENDING}
    else:
        if PREPARED_ARCHIVE is None:
            import re
            pattern = re.compile(re.escape(DATASET_NAME) + r'-[0-9a-f]{64}-.+\.zip')
            candidates = sorted(p for p in (PREPARATION_DIR / 'transport').glob('*.zip')
                                if pattern.fullmatch(p.name))
            if len(candidates) != 1:
                raise ValueError(f'Expected one prepared ZIP for {DATASET_NAME}; found {len(candidates)}. Set PREPARED_ARCHIVE explicitly: {candidates}')
            PREPARED_ARCHIVE = candidates[0]
        receipt = json.loads(Path(str(PREPARED_ARCHIVE) + '.receipt.json').read_text(encoding='utf-8'))
    if receipt.get('kind') != 'm2-prepared-transport-1.0' or receipt.get('remote_verified') is not True:
        raise ValueError('Expected a remotely verified prepared transport receipt.')
    if receipt.get('papers_pending', 0) and not ALLOW_PARTIAL_PILOT:
        raise ValueError('Prepared transport is a partial pilot; set ALLOW_PARTIAL_PILOT=True to opt in.')
    if not RAW_CORPUS_ROOT.exists():
        RAW_CORPUS_ROOT = stage_prepared_archive(PREPARED_ARCHIVE, RAW_CORPUS_ROOT,
                                                 expected_sha256=receipt['sha256'])
    else:
        print('Reusing local raw; parser will verify the entire corpus again:', RAW_CORPUS_ROOT)
    completion = json.loads((RAW_CORPUS_ROOT / 'preparation.json').read_text(encoding='utf-8'))
    if completion.get('snapshot_id') != receipt['snapshot_id']:
        raise ValueError('Archive snapshot mismatch.')
    actual_pending = completion.get('batch', {}).get('papers_pending', 0)
    if receipt.get('papers_pending', actual_pending) != actual_pending:
        raise ValueError('Archive pending-paper count differs from receipt.')
    if actual_pending and not ALLOW_PARTIAL_PILOT:
        raise ValueError('Prepared archive is a partial pilot; set ALLOW_PARTIAL_PILOT=True to opt in.')
for archive in RESTORE_ARCHIVES:
    archive = Path(archive)
    receipt = json.loads(Path(str(archive) + '.receipt.json').read_text(encoding='utf-8'))
    restored = restore_run_batch(archive, OUTPUT_ROOT / ('restore-' + archive.stem),
                                expected_sha256=receipt['sha256'])
    print('Restored, awaiting current-run cache verification:', restored)
if RAW_CORPUS_ROOT.is_dir():
    import csv
    inventory = RAW_CORPUS_ROOT / 'inventory.csv'
    if inventory.is_file():
        with inventory.open(encoding='utf-8', newline='') as stream:
            from itertools import islice
            print('Inventory preview (not validation):', list(islice(csv.DictReader(stream), 5)))
else:
    print('No local raw selected yet; stage/select prepared input before opting in.')


## Run, audit and measure

Select DATASET_NAME (for example Danh, Danh1 or Danh2) and PAPERS_THIS_RUN. M2 automatically selects the unique matching ZIP under preparationM2/transport and stages it locally; preparation is never rerun. Multiple ZIPs require an explicit PREPARED_ARCHIVE. Ordinary visits reconstruct progress from verified backups, retry pending publications first (once per visit), then take unseen publications up to the requested count. Four failed durable attempts move a publication to review_later.csv for later review; successful publications are skipped. Input/config/code/review changes start a separate signature history. Use one runtime per dataset. Queue exhaustion is not M2 acceptance. Backup defaults on and writes under Milestone2/<dataset>/<run_id>/. Keep every ZIP and matching receipt, including metadata-only final batches. The archive contains a batch manifest for restore and run_manifest.json for complete coverage/status; audits, profiles, metrics and environment are also retained, including failed publications. Keep preparationM2 separately for M3 raw inputs and M1 metadata.

Use RUN_PARSER for ordinary execution. One backup worker saves a completed batch while the parser processes the next batch. If the worker is still busy at the next batch boundary, the parser waits. A batch is durable after its Drive ZIP has been verified and its receipt saved. The run waits for final backup and reports failures. RESUME verifies current inputs before looking for cache.

Enable RUN_BENCHMARK to measure the same fresh full parse that produces the M3 outputs. Both RUN_PARSER and RUN_BENCHMARK may be True. Set PAPERS_THIS_RUN=None, choose a new OUTPUT_ROOT, and leave RESUME=False and RESTORE_ARCHIVES empty. On Colab select CPU/accelerator None and record `COLAB_ACCELERATOR_OBSERVED = 'None'`. Without runtime evidence the receipt stays local/unverified.

With BACKUP_ENABLED, completed batches are saved while parsing continues. Wall time and sampled RSS include background backup and its final wait. Setup, input staging, benchmark report generation and its upload are outside measurement. The last backup adds the receipt and samples to the same run on Drive and retries any working outputs without verified backups. Progress prints every 50 completed publications and at the end.


In [ ]:
if RUN_PARSER or RUN_BENCHMARK:
    def report_progress(message):
        print(message, flush=True)

    if CODE_REVISION is None:
        CODE_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
    config = ParseConfig(strict=STRICT, code_revision=CODE_REVISION,
                         reference_review_path=REFERENCE_REVIEW_PATH,
                         ram_sample_interval_seconds=RAM_SAMPLE_INTERVAL_SECONDS)
    if RUN_BENCHMARK:
        if PAPERS_THIS_RUN is not None:
            raise ValueError('Fresh full-corpus benchmark: set PAPERS_THIS_RUN=None; ordinary visits use RUN_PARSER.')
        if RESUME or RESTORE_ARCHIVES:
            raise ValueError('Fresh benchmark cannot resume or restore parsed caches.')
        if COLAB_ACCELERATOR_OBSERVED is not None:
            os.environ['M2_COLAB_ACCELERATOR'] = COLAB_ACCELERATOR_OBSERVED
        receipt = benchmark_run(RAW_CORPUS_ROOT, OUTPUT_ROOT, config,
                                backup_root=DRIVE_BACKUP_ROOT if BACKUP_ENABLED else None,
                                batch_size=BACKUP_BATCH_SIZE, progress=report_progress)
        run_root = OUTPUT_ROOT / receipt.run_id
        print('Benchmark receipt:', run_root / 'benchmark/receipt.json')
        print('Publications:', len(receipt.publications), '| Wall seconds:', round(receipt.wall_seconds, 2),
              '| Peak RSS bytes:', receipt.observed_peak_rss_bytes, '| Output bytes:', receipt.final_output_bytes)
        print('CPU Colab evidence:', receipt.cpu_only_colab, '| Run status:', receipt.environment['run_status'])
        print('Measurement:', receipt.environment['drive_backup'])
        if receipt.environment['backup_errors'] != '0':
            print('Backup errors reported during the run:', run_root / 'run_errors.jsonl')
        if BACKUP_ENABLED:
            print('[Drive] Đang lưu kết quả benchmark.', flush=True)
            benchmark_archive = backup_run_batch(run_root, DRIVE_BACKUP_ROOT)
            if benchmark_archive is not None:
                print(f'[Drive] Đã lưu và xác minh benchmark · {benchmark_archive.name}', flush=True)
    elif BACKUP_ENABLED:
        result = run_with_backup(RAW_CORPUS_ROOT, OUTPUT_ROOT, config, DRIVE_BACKUP_ROOT,
                                 resume=RESUME, batch_size=BACKUP_BATCH_SIZE, max_publications=PAPERS_THIS_RUN,
                                 progress=report_progress)
    else:
        if PAPERS_THIS_RUN is not None:
            raise ValueError('Automatic bounded visits require BACKUP_ENABLED=True. Set PAPERS_THIS_RUN=None for an unbounded local check.')
        result = run(RAW_CORPUS_ROOT, OUTPUT_ROOT, config, resume=RESUME, progress=report_progress)
    if result is not None:
        run_root = result.manifest_path.parent
        from collections import Counter
        print('Processed publications:', result.processed_publications)
        print('Diagnostic counts by code:', dict(Counter(d.code for d in result.diagnostics)))
        queue_path = result.manifest_path.parent / 'queue.json'
        if queue_path.is_file():
            queue = json.loads(queue_path.read_text(encoding='utf-8'))
            print('Dataset queue (whole snapshot):', queue['status_counts'])
            print('Eligible for next visit:', queue['remaining_eligible'])
            print('Deferred after 4 failed attempts:', result.manifest_path.parent / 'review_later.csv')
            print('Progress counts only after verified backup; a visit success is not whole-dataset acceptance.')
        print('Manifest:', result.manifest_path)
        manifest = json.loads(result.manifest_path.read_text(encoding='utf-8'))
        print('Run/snapshot/status:', manifest['run_id'], manifest['snapshot_id'], manifest['status'])
        print('Metrics:', result.metrics_path)
        if result.metrics_path.is_file():
            measured = json.loads(result.metrics_path.read_text(encoding='utf-8'))
            print('Version status counts:', measured['summary']['status_counts'])
        for record in manifest['publications'][:3]:
            print('Publication audit:', {key: record[key] for key in ('arxiv_id', 'expected_versions', 'export_ready', 'working_path', 'published_path', 'audit_path', 'used_cache')})
            if record['working_path']:
                publication = read_publication(result.manifest_path.parent / record['working_path'])
                print('Read-back states/root map:', publication.version_states, publication.root_ids)
        if result.status != 'success':
            print('Not complete: inspect all diagnostics, blocked versions and failed writes.')
else:
    print('Not run. No parse result or benchmark numbers are produced by this smoke execution.')


## Publish tables for Milestone 3

The cell below reads local M2 output and prepared M1 metadata after the benchmark. It publishes five Parquet datasets and a manifest under the same M2 run. Only export-ready publications are included. For excluded publications, the manifest reports M1 source evidence and M2 status and error codes by version. If an upload fails while the local run is still available, rerun this cell by itself. Keep all original backup ZIPs and receipts.


In [ ]:
if EXPORT_HANDOFF_TABLES and run_root is not None:
    from time import perf_counter
    from parsing.handoff import export_handoff_tables

    handoff_started = perf_counter()
    print('[M3] Đang xuất bảng Parquet.', flush=True)
    handoff = export_handoff_tables(RAW_CORPUS_ROOT, run_root,
                                   backup_root=DRIVE_BACKUP_ROOT if BACKUP_ENABLED else None)
    handoff_manifest = json.loads((handoff / 'manifest.json').read_text(encoding='utf-8'))
    print('[M3] Đã lưu bảng tại:', handoff, flush=True)
    print(f"[M3] {handoff_manifest['publications_included']} bài được bàn giao · "
          f"{handoff_manifest['publications_excluded']} bài bị loại · "
          f"Xuất và xác minh {perf_counter() - handoff_started:.1f} giây (ngoài benchmark)")
    print('[M3] Lý do theo version của các bài bị loại:', handoff_manifest['excluded_version_counts'])


## Acceptance and handoff (not automatically signed)

- Real multi-file publication with >=2 versions: raw → tree → normalized tree → three tables and unified bib. Compare citations against bibtex; inspect math, table cells and captions for meaning.
- Unchanged sentence moved to another parent: one element and all edges; repeated occurrences retain order. Changed paragraph body/reversed items produce different component IDs.
- Open JSON then read back: authors lists, null year/URL, tuple-key map rows, integer version/root maps and final statuses. All-no_tex is covered without fake roots; blocked/partial never become success.
- Independently alter source/config/code/review and rerun; stale cache must not be used. Interrupt/fail writes in fixtures; previous good output remains available.
- Pilot/profile 50–100 diverse real papers, compare all tables/occurrences/cites/status/coverage and p50/p95/RAM before considering workers. Stage timings are in profile/*.json; no speed promise without measurement.
- On fresh CPU Colab, calculate wall/mean/output bytes for one sample from raw receipts; look at CPU settings. Local smoke is not Colab evidence.
- Book acceptance needs metrics from the intended real run and human review: clean execute/build, HTML/Pages search/status filter/empty/reset, runtime hover/zoom/reset/run filter. Store build log, URL, commit and human checklist outside code Git.
- Khánh signs coverage/pipeline; Khang signs normalization/tree; Nguyên signs bib/cites. Do not fill reviewer names as approvals or claim M2 complete while these checks are pending. Handoff follows assignment section 12.
